## Goal: Finalize analysis to answer our 3 supporting questions
- Do fires cross wetland boundaries? (yes, supported by fact that there is wetland area burned each year)
    - 
- Are wildfires less frequent near wetlands? 
    - Goal: use chi squared test to see if burned area inside wetlands is about what you would expect for average of all burned area or if it is statistically different

## Set up all the data

In [1]:
# import packages

import platform
from pathlib import Path
import pandas as pd
import geopandas as gpd
import seaborn as sns
import calendar
import matplotlib.pyplot as plt
import pygris
from shapely.geometry import Point
import numpy as np
from scipy.stats import chi2_contingency, fisher_exact


In [2]:
# import data 

# filepath

project_fldr = Path.cwd().parents[1]
raw_fldr = project_fldr / 'data' / 'raw'
processed_fldr = project_fldr / 'data' / 'processed'

# #  fire data by year
fire_2020 = gpd.read_file(processed_fldr / "fire_explore/fire_2020.geojson")
fire_2021 = gpd.read_file(processed_fldr / "fire_explore/fire_2021.geojson")
fire_2022 = gpd.read_file(processed_fldr / "fire_explore/fire_2022.geojson")
fire_2023 = gpd.read_file(processed_fldr / "fire_explore/fire_2023.geojson")
fire_2024 = gpd.read_file(processed_fldr / "fire_explore/fire_2024.geojson")
fire_2025 = gpd.read_file(processed_fldr / "fire_explore/fire_2025.geojson")

# california outline
ca_outline = gpd.read_file(processed_fldr /"fire_explore/ca_outline.geojson")


# set a path to the main box folder based on machine's os
# if windows: 
if platform.system() == 'Windows':
    box_path = Path.home() / 'Box'
# if mac:
else:
    box_path = Path.home() / "Library" / "CloudStorage" / "Box-Box"

#Confirm that the box path is valid
if box_path.exists():
    print(f'Box folder found at {box_path}')
else:
    print(f'Failed to locate the Box folder.\nIs the BoxDrive app running?\nAre you logged in?')

# create path object that goes to the specific folder on Box drive
wetlands_path = box_path / 'ENVIRON 712L-Students' / 'Data' / 'BunkerAndersonTseng' / 'wetlands_subset' / 'wetlands_subset.geojson'
print(wetlands_path.exists())

# read in the wetlands geojson file, this will take a WHILE - 10 mins 1st try 
wetlands = gpd.read_file(wetlands_path, engine="pyogrio")



Box folder found at /Users/katanderson/Library/CloudStorage/Box-Box
True


In [3]:
# convert CRS function
def crs_convert(shp):
    shp = shp.to_crs(epsg=26910)
    return shp

# check CRS of files
print(fire_2020.crs)
print(ca_outline.crs)

EPSG:4326
EPSG:4326


In [4]:
# convert all shp CRS
fire_2020 = crs_convert(fire_2020)
fire_2021 = crs_convert(fire_2021)
fire_2022 = crs_convert(fire_2022)
fire_2023 = crs_convert(fire_2023)
fire_2024 = crs_convert(fire_2024)
fire_2025 = crs_convert(fire_2025)
ca_outline = crs_convert(ca_outline)
wetlands = crs_convert(wetlands)

In [5]:
# Generate 10,000 random points
n_pts = 10000

# Extract California polygon
study_area = ca_outline.geometry.iloc[0]

# Generate random points
random_pts = gpd.GeoSeries(
    [study_area],
    crs=ca_outline.crs
).sample_points( # take a random sample
    size=n_pts,
    method="uniform",
    seed=42 # but get the same random pts every time i run this code
).explode(index_parts=False).reset_index(drop=True) # separate each pt into its own line

# Convert to GeoDataFrame
random_pts = gpd.GeoDataFrame(
    geometry=random_pts,
    crs=ca_outline.crs
)


In [6]:
# place fires in dict for easy access
fires = {
    2020: fire_2020,
    2021: fire_2021,
    2022: fire_2022,
    2023: fire_2023,
    2024: fire_2024,
    2025: fire_2025
}

## Use random points to determine correlation between wetlands and fire
- Question: is land inside wetlands less likely to burn?
- Results appear not statistically significant, so disregard this analysis for now

In [7]:
# function for chi squared analysis,  Y/N wetlands and Y/N fire
# can give us statistical answer to "are wetlands less likely to burn"
def classify_pts(fire):
    # create copy of random points
    points = random_pts.copy().reset_index(drop=True)

    # create a copy of wetlands
    wetlands_cpy = wetlands.copy().reset_index(drop=True)

    # identify wetland intersections
    wetland_join = gpd.sjoin(
        points, 
        wetlands_cpy,
        how='left'
    )
    
    # classify wetland presence (1=Y, 0=N)
    points['wetland'] = ( # add column called "wetland" to points 
        wetland_join['index_right'] # creates column that shows corresponding wetland polygon to point if any
        .notna() # creates T/F column where T = intersects (has a number)
        .groupby(level=0) # combines any points that intersect with multiple wetland shapefiles
        .any() # if a pt overlaps with at least 1 wetland, it is T
        .astype(int) # changes T/F boolean to 1/0 values 

    ) 

    # identify fire intersections
    fire_join = gpd.sjoin(
            points, 
            fire,
            how='left'
        )
    
    # classify fire presence
    points["fire"] = (
        fire_join["index_right"]
        .notna()
        .groupby(level=0)
        .any()
        .astype(int)
    )

    return points



In [8]:
# run function on each year of shapefile
results = {}
for year, fire in fires.items():
    results[year] = classify_pts(fire)

In [9]:
# generate contingency table for each year
tables = {}
for year, points in results.items():
    tables[year] = pd.crosstab(
        points['wetland'],
        points['fire']
    )

In [ ]:
# use fischer's exact test to determine if there is any statistical relationship 
# null hypothesis: there is no relationship between wetland occurance and fire occurance

# test for 2020 
odds_ratio, p_fisher = fisher_exact(tables[2020])

print("Odds ratio:", odds_ratio) # odds ratio below 1 = wetland locations had lower observed odds of burning than non-wetland locations.
print("Fisher's exact p-value:", p_fisher) # p value below 0.05 indicates statistical significance

Odds ratio: 0.3643067930471766
Fisher's exact p-value: 0.00439020042980919


In [11]:
# run fischer's exact test for every year of data

# store Fisher's exact test results
fisher_results = {}

for year, table in tables.items():

    # run Fisher's exact test
    odds_ratio, p_value = fisher_exact(table)

    # store results
    fisher_results[year] = {
        "Odds Ratio": odds_ratio,
        "P-value": p_value,
        "Significant": p_value < 0.05
    }

In [12]:
# convert results into a dataframe
fisher_df = pd.DataFrame.from_dict(
    fisher_results,
    orient="index"
)

fisher_df.index.name = "Year"

print(fisher_df)

      Odds Ratio   P-value  Significant
Year                                   
2020    0.364307  0.004390         True
2021    0.557974  0.137337        False
2022    2.239427  0.243439        False
2023    0.642303  1.000000        False
2024    0.217046  0.134483        False
2025    0.000000  0.268162        False


## Use random points to help determine if land near wetlands is less likely to burn
- Is land closer to wetlands less likely to burn?

In [ ]:

import time

# goal: generate random points, and classify their distance to a wetland and presence of fire (t/f)
# to answer Q "is land closer to wetlands less likely to burn?"

def classify_dist(fire):

    total_start = time.perf_counter()

    # create copy of random points
    step_start = time.perf_counter()
    points = random_pts.copy().reset_index(drop=True)
    print(f"Copy random points: {time.perf_counter() - step_start:.2f} seconds")

    # create a copy of wetlands
    step_start = time.perf_counter()
    wetlands_cpy = wetlands.copy().reset_index(drop=True)
    print(f"Copy wetlands: {time.perf_counter() - step_start:.2f} seconds")

    # calculate distance to nearest wetland
    step_start = time.perf_counter()
    nearest_wetlands = gpd.sjoin_nearest(
        points,
        wetlands_cpy,
        how="left",
        distance_col="wetland_distance"
    )

    # keep one distance per random point (including tied nearest wetlands)
    nearest_distances = (
        nearest_wetlands
        .groupby(level=0)["wetland_distance"]
        .min()
    )

    points["wetland_distance"] = nearest_distances.reindex(points.index)

    print(f"Calculate nearest wetland distances: {time.perf_counter() - step_start:.2f} seconds")

    # classify distance from wetlands
    step_start = time.perf_counter()
    points["distance_category"] = pd.cut(
        points["wetland_distance"],
        bins=[-1, 1000, 5000, 10000, 20000, float("inf")],
        labels=["0-1 km", "1-5 km", "5-10 km",
                "10-20 km", "Over 20 km"]
    )
    print(f"Classify distances: {time.perf_counter() - step_start:.2f} seconds")

    # identify wetland intersections
    step_start = time.perf_counter()
    wetland_join = gpd.sjoin(
        points,
        wetlands_cpy,
        how="left",
        predicate="intersects"
    )
    print(f"Wetland spatial join: {time.perf_counter() - step_start:.2f} seconds")

    # classify wetland presence (1=Y, 0=N)
    points["wetland"] = (
        wetland_join["index_right"]
        .notna()
        .groupby(level=0)
        .any()
        .astype(int)
    )

    # identify fire intersections
    step_start = time.perf_counter()
    fire_join = gpd.sjoin(
        points,
        fire,
        how="left",
        predicate="intersects"
    )
    print(f"Fire spatial join: {time.perf_counter() - step_start:.2f} seconds")

    # classify fire presence
    points["fire"] = (
        fire_join["index_right"]
        .notna()
        .groupby(level=0)
        .any()
        .astype(int)
    )

    print(f"TOTAL FUNCTION TIME: {time.perf_counter() - total_start:.2f} seconds")

    return points



In [ ]:
# test our distance function
analysis_2020 = classify_dist(fire_2020)

print(analysis_2020[
    ["wetland_distance", "distance_category", "fire"]
].head(10))

Copy random points: 0.00 seconds
Copy wetlands: 0.02 seconds
